In [1]:
import ollama

## **Model**

In [2]:
MODEL = "llama3.2:1b"
MAX_STEPS = 5

## **Tool**

Composed of three parts: 
- doc string
- type hints eg "city"
- name

In [3]:
def get_weather(city:str) -> str:
    """ Get current weather for a given city"""
    return f"22°C and light rain experienced in {city} "

In [4]:
TOOLS = {
    "get_weather":get_weather,
}

In [5]:
print(TOOLS)

{'get_weather': <function get_weather at 0x7feaf5900400>}


## **Agent Loop**

COmposed of loop with maximum nuumber of steps, this acts as a guardrail to limit the number of calls made by the model

In [6]:
def run_agent(user_question: str, max_steps=MAX_STEPS):
    # The conversation history. The model has no memory,
    # so we send the whole history on every call.
    messages = [
        {"role": "system", "content": "You can use tools. After you get a tool result, use it to answer the user."},
        {"role": "user", "content": user_question},
    ]

    for step in range(max_steps):
        # 1. Ask the model what to do next
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=list(TOOLS.values()),
        )

        # 2. Save the model's reply in the history
        messages.append(response.message)

        # 3. No tool requested? Then this is the final answer -> stop
        if not response.message.tool_calls:
            return response.message.content

        # 4. The model asked for tool(s): run each one
        for call in response.message.tool_calls:
            name = call.function.name          # e.g. "get_weather"
            args = call.function.arguments    # e.g. {"city": "Nairobi"}

            # Small models sometimes wrap the real arguments inside "parameters",
            # like a box inside a box. Keep opening until we reach the real ones.
            while "parameters" in args:
                args = args["parameters"]

            print(f"step {step + 1}, model requested: {name}{args}")

            try:
                result = TOOLS[name](**args)   # run the real Python function
            except Exception as e:
                result = f"ERROR: {e}"         # send errors back instead of crashing

            print(f"   tool returned: {result}")

            # 5. Give the tool's result back to the model
            messages.append({"role": "tool", "content": str(result), "tool_name": name})

    return f"Stopped: reached maximum number of steps ({max_steps})"


In [7]:
   # Needs a tool -> watch the loop run.
print(run_agent("Should I carry an umbrella in Nairobi today?"))
print("-" * 60)

step 1, model requested: get_weather{'city': 'Nairobi'}
   tool returned: 22°C and light rain experienced in Nairobi 
It seems that it is a good idea to carry an umbrella in Nairobi today, as there is a chance of light rain. The weather forecast indicates a high chance of light rain, making it a wise decision to bring an umbrella for added protection.
------------------------------------------------------------
